In [23]:
import polars as pl
import numpy as np
import sys
sys.path.append("..")
from src.cleaning.limpiado import limpiar_datos
from config.rutas import RUTA_DATOS_CRUDOS

df_limpio = limpiar_datos(RUTA_DATOS_CRUDOS)
print("Filas sin nulos en edad:", df_limpio.shape[0])

Filas sin nulos en edad: 24560


In [24]:
#df_limpio = df_limpio.filter(
#    (pl.col("edad_primer_union") >= 9) & 
#    (pl.col("edad_primer_union") <= 97)
#)

medidas_simples = df_limpio.select([
    pl.col("edad_primer_union").mean().alias("Media_Edad"),
    pl.col("edad_primer_union").median().alias("Mediana_Edad"),
    pl.col("edad_primer_union").mode().first().alias("Moda_Edad"),
    pl.col("edad_primer_union").quantile(0.25).alias("Q1_Edad"),
    pl.col("edad_primer_union").quantile(0.75).alias("Q3_Edad"),
    pl.col("edad_primer_union").quantile(0.10).alias("P10_Edad"),
    pl.col("edad_primer_union").quantile(0.90).alias("P90_Edad"),
    
    pl.col("num_hijos").mean().alias("Media_Hijos"),
    pl.col("num_hijos").median().alias("Mediana_Hijos"),
    pl.col("num_hijos").mode().first().alias("Moda_Hijos"),
    pl.col("num_hijos").quantile(0.25).alias("Q1_Hijos"),
    pl.col("num_hijos").quantile(0.75).alias("Q3_Hijos"),
    pl.col("num_hijos").quantile(0.10).alias("P10_Hijos"),
    pl.col("num_hijos").quantile(0.90).alias("P90_Hijos"),
    pl.col("nivel_escolaridad").mode().alias("moda_escolaridad")
])

medidas_simples

Media_Edad,Mediana_Edad,Moda_Edad,Q1_Edad,Q3_Edad,P10_Edad,P90_Edad,Media_Hijos,Mediana_Hijos,Moda_Hijos,Q1_Hijos,Q3_Hijos,P10_Hijos,P90_Hijos,moda_escolaridad
f64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str
13.198453,7.0,0,2.0,17.0,0.0,30.0,2.517671,3.0,3,1.0,3.0,1.0,4.0,"""B1"""


In [25]:
edades = df_limpio["edad_primer_union"].to_numpy()
hijos = df_limpio["num_hijos"].to_numpy()
pesos = df_limpio["factor_expansion"].to_numpy()

media_edad_ponderada = np.average(edades, weights=pesos)
media_hijos_ponderada = np.average(hijos, weights=pesos)

print(f"Media SIMPLE de Edad:    {medidas_simples['Media_Edad'][0]:.2f} años")
print(f"Media PONDERADA de Edad: {media_edad_ponderada:.2f} años")
print("-" * 40)
print(f"Media SIMPLE de Hijos:    {medidas_simples['Media_Hijos'][0]:.2f}")
print(f"Media PONDERADA de Hijos: {media_hijos_ponderada:.2f}")

Media SIMPLE de Edad:    13.20 años
Media PONDERADA de Edad: 13.06 años
----------------------------------------
Media SIMPLE de Hijos:    2.52
Media PONDERADA de Hijos: 2.45


In [26]:
medidas_variabilidad = df_limpio.group_by("sufrio_violencia_pareja").agg([
    
    # Medidas para edad en la primera unión
    (pl.col("edad_primer_union").max() - pl.col("edad_primer_union").min()).alias("Rango_Edad"), #Rango (Máximo - Mínimo)
    pl.col("edad_primer_union").var().alias("Varianza_Edad"),
    pl.col("edad_primer_union").std().alias("Desv_Std_Edad"),
    ((pl.col("edad_primer_union").std() / pl.col("edad_primer_union").mean()) * 100).alias("CV_Edad_%"),
    (pl.col("edad_primer_union").quantile(0.75) - pl.col("edad_primer_union").quantile(0.25)).alias("IQR_Edad"),

    # medidas para numeros de hijos
    (pl.col("num_hijos").max() - pl.col("num_hijos").min()).alias("Rango_Hijos"),
    pl.col("num_hijos").var().alias("Varianza_Hijos"),
    pl.col("num_hijos").std().alias("Desv_Std_Hijos"),
    ((pl.col("num_hijos").std() / pl.col("num_hijos").mean()) * 100).alias("CV_Hijos_%"),
    (pl.col("num_hijos").quantile(0.75) - pl.col("num_hijos").quantile(0.25)).alias("IQR_Hijos")

])
#0=No reportó violencia de pareja
#1=Reportó violencia de pareja
medidas_variabilidad

sufrio_violencia_pareja,Rango_Edad,Varianza_Edad,Desv_Std_Edad,CV_Edad_%,IQR_Edad,Rango_Hijos,Varianza_Hijos,Desv_Std_Hijos,CV_Hijos_%,IQR_Hijos
i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64
0,99,411.096307,20.27551,149.624759,15.0,3,1.37973,1.174619,46.412341,2.0
1,99,324.57527,18.015973,143.46354,14.0,3,1.491385,1.221223,48.971376,3.0
